Importowanie bibliotek

In [ ]:

#biblioteka do obliczen numerycznych
import numpy as np
#biblioteka do rysowania wykresow 2D
import matplotlib.pyplot as plt
#animacje
from matplotlib.animation import FuncAnimation
#obliczenia symboliczne
import sympy as sp
#wyswietlenie animacji notebook
from IPython.display import HTML

Zadanie na dst/dst+: Podstawowy gradient

In [ ]:
# Zwracanie wartości funkcji w punkt x,y
def funkcja(x, y):
  return y * np.sin(x) - x * np.cos(y)

In [ ]:
def gradient_funkcja(x, y):
  #pochodna po x
  pochodna_x = y * np.cos(x) - np.cos(y)
  #pochodna po y
  pochodna_y = x * np.sin(x) + x * np.sin(y)
  #Zwracanie wartości wektora gradientu pochodnych w punkt x,y
  return np.array([pochodna_x, pochodna_y])

In [ ]:
#Parametry
#learning rate - dlugosc kroku w kierunek przeciwny do gradient
lr = 0.05
# liczba iteracji
n_iter = 100
# punkt startowy - x0 i y0 (domyślnie float)
start = np.array([2.0, 2.0])



#utworzenie kopii, by nienadpisywac zmiennej start
x, y = start.copy()
# lista punktów
points = [start]
for _ in range(n_iter):
  #gradient w biezacy punkt
    gradient = gradient_funkcja(x, y)
    # krok przeciwny do gradient - jednoczesne aktualizowanie x i y
    x, y = np.array([x, y]) - lr * gradient
    #dopisanie nowego punktu
    points.append([x, y])

points = np.array(points)

In [ ]:
# 3 różne pkt startowe
start_points = [(-5, -1), (4, 2), (1.5, -2)]
paths = []
for start in start_points:
    x, y = start
    path = [(x, y)]
    for _ in range(n_iter):
        gradient = gradient_funkcja(x, y)
        x, y = np.array([x, y]) - lr * gradient
        path.append((x, y))
    paths.append(np.array(path))



Wykres poziomic

In [ ]:
# zakres osi x
x_vals = np.linspace(-10, 10, 400)
#zakres osi y
y_vals = np.linspace(-10, 10, 400)
#siatka współrzednych - macierze o rozmiar 400x400
X, Y = np.meshgrid(x_vals, y_vals)
#wartosci funkcji na siatce
Z = funkcja(X, Y)

#obiekt figury i osi
fig, ax = plt.subplots(figsize=(7, 5))
#ustawiłam 50 poziomic
contours = ax.contour(X, Y, Z, levels=50, cmap='plasma')
#podpisanie poziomic
ax.clabel(contours, inline=True, fontsize=8)
#tytul wykresu poziomic
ax.set_title("Gradient Descent dla f(x,y)=y*sin(x)-x*cos(y)")
#etykieta osi x
ax.set_xlabel("x")
#etykieta osi y
ax.set_ylabel("y")
for path in paths:
  #osobne łamane z kropkami
    ax.plot(path[:, 0], path[:, 1], marker='o', markersize=3)

# Obliczam wartości funkcji w punktach końcowych
final_values = [funkcja(path[-1, 0], path[-1, 1]) for path in paths]
best_idx = np.argmin(final_values)
worst_idx = np.argmax(final_values)

# Pobieram współrzędne najlepszego i najgorszego rozwiązania
best_point = paths[best_idx][-1]
worst_point = paths[worst_idx][-1]

# Zaznaczam te punkty na wykresie
ax.scatter(best_point[0], best_point[1], color='lime', s=100, label='Najlepsze rozwiązanie')
ax.scatter(worst_point[0], worst_point[1], color='red', s=100, label='Najgorsze rozwiązanie')

# Dodaje legendę
ax.legend()

    #wyswietlanie wykres
plt.show()

Zadanie na db: Równania funkcji pochodnych z wykorzystaniem biblioteki sympy

In [ ]:
#definicja symboli
x_symbole, y_symbole = sp.symbols('x y')
#wyrazenie symboliczne
f_symbole = y_symbole * sp.sin(x_symbole) - x_symbole * sp.cos(y_symbole)
#liczenie symbolicznie poch. czastkowe
grad_symbole = [sp.diff(f_symbole, x_symbole), sp.diff(f_symbole, y_symbole)]
print("Symboliczny gradient f(x,y) = y*sin(x) - x*cos(y):")
print("∂f/∂x =", grad_symbole[0])
print("∂f/∂y =", grad_symbole[1])

Zadanie na db + : Animacja

In [ ]:
#Poczekać aż się załaduje i kiknąć play:)
fig, ax = plt.subplots(figsize=(7, 5))
contours = ax.contour(X, Y, Z, levels=50, cmap='coolwarm')
ax.clabel(contours, inline=True, fontsize=8)
ax.set_title("Animacja Gradient Descent")
ax.set_xlabel("x")
ax.set_ylabel("y")

line, = ax.plot([], [], 'r-', lw=2)
point, = ax.plot([], [], 'ro')

def init():
    # inicjalizacja pustymi sekwencjami
    line.set_data([], [])
    point.set_data([], [])
    return line, point

def update(i):
    # i jest indeksem klatki
    # pokażemy ścieżkę do i włącznie, dlatego [:i+1]
    line.set_data(points[:i+1, 0], points[:i+1, 1])
    point.set_data([points[i, 0]], [points[i, 1]])  # pojedynczy punkt jako listy
    return line, point

#tworzenie animacji
anim = FuncAnimation(
    fig, update,
    # nie n_iter bo mamy o 1 punkt więcej (start)
    frames=len(points),
    init_func=init,
    #odstp miedzy klatkami w milisekundy
    interval=150,
    blit=False
)

#  Render bezpośrednio w komórce (JS/HTML)
HTML(anim.to_jshtml())



Zadanie na bdb: Odmiana gradientu - Momentum

In [ ]:
#start - punkt poczatkowy, beta - wspolczynnik momentum, steps - liczba iteracji
def gradient_descent_momentum(start, lr=0.05, beta=0.7, steps=80):
    x, y = start
    v = np.zeros(2)
    path = [(x, y)]
    for _ in range(steps):
        gradient = gradient_funkcja(x, y)
        v = beta * v + lr * gradient
        x, y = np.array([x, y]) - v
        path.append((x, y))
    return np.array(path)

In [ ]:
#liczenie sciezek momentum
paths_momentum = [gradient_descent_momentum(p) for p in start_points]
# wykres porownawczy gradient descent(czerwone), momentum(niebieski) na poziomicach
fig, ax = plt.subplots(figsize=(8, 6))
contours = ax.contour(X, Y, Z, levels=40, cmap='plasma')
ax.clabel(contours, inline=True, fontsize=8)
ax.set_title("Porównanie Gradient Descent i Momentum")
ax.set_xlabel("x")
ax.set_ylabel("y")
#rysowanie trajektorii gradient descent
for i, p in enumerate(paths):
    ax.plot(p[:, 0], p[:, 1], 'r-', label='Gradient Descent' if i == 0 else "")
    #rysowanie trajektorii momentum
for i, p in enumerate(paths_momentum):
    ax.plot(p[:, 0], p[:, 1], 'b-', label='Momentum' if i == 0 else "")

ax.legend()
plt.show()


Wyniki

In [ ]:
print("\n Wyniki końcowe")
for i, start in enumerate(start_points):
  print(f"Punkt startowy {start}:")
  print(f"Minimum = Gradient Descent): {paths[i][-1]}")
  print(f"Minimum = Momentum: {paths_momentum[i][-1]}")
  print()